# Differential Equations — Session 33
## Section 7.4: Operational Properties II

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives

Students should be able to use derivatives of transforms, define convolution, apply the convolution theorem, transform integrals, solve simple Volterra equations, and compute transforms of periodic functions.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Core 90-minute path

| Time | Topic |
|---:|---|
| 0–18 min | Derivatives of transforms |
| 18–45 min | Convolution definition and geometry |
| 45–62 min | Convolution theorem |
| 62–76 min | Integral equations |
| 76–88 min | Periodic functions |
| 88–90 min | Exit check |

The integrodifferential circuit is an optional extension.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import quad, solve_ivp
from scipy.signal import fftconvolve
from scipy.linalg import expm
from IPython.display import display, Markdown

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=6, suppress=True)

def unit_step(t, a=0.0):
    t = np.asarray(t)
    return (t >= a).astype(float)

print("Notebook ready.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)

## Formal theory reference

### Theorem 7.4-A — Derivatives of transforms

If $\mathcal L\{f\}=F$, then

$$
\mathcal L\{t^nf(t)\}
=
(-1)^nF^{(n)}(s).
$$

### Definition 7.4-B — Convolution

$$
(f*g)(t)
=
\int_0^t f(\tau)g(t-\tau)\,d\tau.
$$

### Theorem 7.4-C — Convolution theorem

$$
\mathcal L\{f*g\}
=
F(s)G(s).
$$

Therefore,

$$
\mathcal L^{-1}\{F(s)G(s)\}=f*g.
$$

### Corollary 7.4-D — Transform of an integral

$$
\mathcal L\left\{\int_0^t f(\tau)\,d\tau\right\}
=
\frac{F(s)}{s}.
$$

### Theorem 7.4-E — Periodic function

If $f$ has period $T$, then

$$
\mathcal L\{f\}
=
\frac{\int_0^T e^{-st}f(t)\,dt}
{1-e^{-sT}}.
$$

### Classroom Checkpoint — Convolution Meaning

What does

$$
(f*g)(t)=\int_0^t f(\tau)g(t-\tau)\,d\tau
$$

represent in a linear input–output system?

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. Differentiate in the transform domain

Since

$$
\mathcal L\{\sin bt\}
=
\frac{b}{s^2+b^2},
$$

we obtain

$$
\mathcal L\{t\sin bt\}
=
-\frac{d}{ds}\left(\frac{b}{s^2+b^2}\right)
=
\frac{2bs}{(s^2+b^2)^2}.
$$

In [ ]:
s, b = sp.symbols("s b", positive=True)
F = b/(s**2+b**2)
display(sp.simplify(-sp.diff(F, s)))

## 2. Convolution as overlap

Let $f(t)=e^{-t}$ and $g(t)=1$ for $t\ge0$. Then

$$
(f*g)(t)
=
\int_0^t e^{-\tau}\,d\tau
=
1-e^{-t}.
$$

In [ ]:
def convolution_overlap(t0=3.0):
    tau = np.linspace(0, max(6, t0+1), 700)
    f = np.exp(-tau)
    reflected_shifted = ((tau >= 0) & (tau <= t0)).astype(float)
    product = f*reflected_shifted

    plt.plot(tau, f, label=r"$f(\tau)=e^{-\tau}$")
    plt.plot(tau, reflected_shifted, label=r"$g(t-\tau)$")
    plt.fill_between(tau, product, alpha=0.25, label="overlap product")
    plt.legend()
    plt.title(fr"Convolution integrand at $t={t0:.2f}$")
    plt.show()

    value = np.trapz(product, tau)
    print("numerical convolution:", value)
    print("exact:", 1-np.exp(-t0))

if WIDGETS_AVAILABLE:
    interact(
        convolution_overlap,
        t0=FloatSlider(min=0, max=6, step=0.1, value=3)
    )
else:
    convolution_overlap()

## 3. Dynamic response as convolution

For a zero-state system

$$
y''+2y'+5y=f(t),
$$

the impulse response is

$$
h(t)=\frac12e^{-t}\sin2t.
$$

Then

$$
y=h*f.
$$

In [ ]:
t = np.linspace(0, 20, 3000)
dt = t[1]-t[0]
h = 0.5*np.exp(-t)*np.sin(2*t)
forcing = ((t >= 2) & (t <= 5)).astype(float)
response = fftconvolve(h, forcing)[:len(t)]*dt

plt.plot(t, forcing, label="input")
plt.plot(t, response, label="convolution response")
plt.legend()
plt.title("Pulse response through convolution")
plt.show()

In [ ]:
def pulse_width_response(start=2.0, width=3.0):
    t = np.linspace(0, 20, 3000)
    dt = t[1]-t[0]
    h = 0.5*np.exp(-t)*np.sin(2*t)
    f = ((t >= start) & (t <= start+width)).astype(float)
    y = fftconvolve(h, f)[:len(t)]*dt
    plt.plot(t, f, label="forcing")
    plt.plot(t, y, label="response")
    plt.legend()
    plt.title("Input duration changes the response")
    plt.show()

if WIDGETS_AVAILABLE:
    interact(
        pulse_width_response,
        start=FloatSlider(min=0, max=8, step=0.25, value=2),
        width=FloatSlider(min=0.25, max=8, step=0.25, value=3)
    )
else:
    pulse_width_response()

## 4. Volterra integral equation

Consider

$$
f(t)=1+\int_0^t f(\tau)\,d\tau.
$$

Taking transforms gives

$$
F(s)=\frac1s+\frac{F(s)}s.
$$

Hence

$$
F(s)=\frac1{s-1}
$$

and

$$
f(t)=e^t.
$$

In [ ]:
t = np.linspace(0, 3, 400)
f = np.exp(t)
integral_term = np.array([quad(lambda tau: np.exp(tau), 0, ti)[0] for ti in t])
plt.plot(t, f, label=r"$f(t)$")
plt.plot(t, 1+integral_term, linestyle="--", label=r"$1+\int_0^t f$")
plt.legend()
plt.title("Verification of a Volterra equation")
plt.show()

## 5. Periodic square wave

Let

$$
f(t)=
\begin{cases}
1,&0\le t<1,\\
-1,&1\le t<2,
\end{cases}
$$

and extend periodically with period $2$.

In [ ]:
t = np.linspace(0, 8, 1200)
phase = np.mod(t, 2)
square = np.where(phase < 1, 1, -1)

plt.step(t, square, where="post")
plt.ylim(-1.3, 1.3)
plt.xlabel("t")
plt.ylabel("f(t)")
plt.title("Periodic square wave")
plt.show()

In [ ]:
s = sp.symbols("s", positive=True)
T = 2
numerator = sp.integrate(sp.exp(-s*sp.Symbol("t")), (sp.Symbol("t"), 0, 1)) - \
            sp.integrate(sp.exp(-s*sp.Symbol("t")), (sp.Symbol("t"), 1, 2))
F_periodic = sp.simplify(numerator/(1-sp.exp(-2*s)))
display(F_periodic)

## Optional extension — Integrodifferential circuit

For an LRC circuit written in current form,

$$
Li'+Ri+\frac1C\int_0^t i(\tau)\,d\tau=E(t),
$$

the transform of the integral becomes $I(s)/s$, reducing the equation to algebra.

In [ ]:
# Illustrative response to a finite-duration voltage pulse.
L, R, C = 0.2, 2.0, 0.1
start, stop = 1.0, 4.0

def rhs(t, z):
    i, q = z
    E = 10.0 if start <= t <= stop else 0.0
    return [(E-R*i-q/C)/L, i]

grid = np.linspace(0, 10, 1000)
sol = solve_ivp(rhs, (0, 10), [0, 0], t_eval=grid, rtol=1e-9, atol=1e-11)
forcing = np.where((grid >= start) & (grid <= stop), 10.0, 0.0)

plt.plot(grid, forcing/10, label="scaled voltage")
plt.plot(grid, sol.y[0], label="current")
plt.legend()
plt.title("Continuous current under discontinuous voltage")
plt.show()

## Classroom Checkpoint — Exit Check

Find

$$
\mathcal L\{t\cos 3t\}.
$$

> Pause here. Let students commit to an answer before running the next cell.